In [1]:
import numpy as np

In [ ]:
# Sampling + tensor shape
FS = 100                 # Hz, matches hardware ODR
T_MAX = 4500             # samples = 9 s @ 500 Hz
N_CHANNELS = 30          # 5 IMUs x (accel xyz + gyro xyz)
N_IMUS = 5
ACCEL_AXES = [0, 1, 2]   # relative to each IMU's 6-channel block
GYRO_AXES = [3, 4, 5]

# Per-class parameters. Index = class label.
CLASS_PARAMS = [
    {"accel_std": 0.3, "gyro_std": 0.2, "length": 1500, "freq_hz": 1.0},
    {"accel_std": 0.8, "gyro_std": 0.6, "length": 2500, "freq_hz": 2.0},
    {"accel_std": 1.5, "gyro_std": 1.2, "length": 3800, "freq_hz": 3.0},
]

NOISE_STD = 0.1          # shared Gaussian noise floor across all classes
SINE_AMPLITUDE = 0.5     # amplitude of the class-specific sinusoidal component

In [3]:
"""
Synthetic IMU data generator for APEX pipeline validation.

Purpose: produce a 3-class dataset where the class-separating features are
known in advance, so that end-to-end pipeline correctness (ingestion -> model
-> evaluation) can be verified before real climbing data exists.

Class design (difficulty ordinal: 0=easy, 1=medium, 2=hard):
    - accel std       : 0.3  / 0.8  / 1.5
    - gyro std        : 0.2  / 0.6  / 1.2
    - attempt length  : 1500 / 2500 / 3800 samples (rest zero-padded to 4500)
    - dominant freq   : 1 Hz / 2 Hz / 3 Hz sinusoidal component

All 30 channels (5 IMUs x 6 axes) share the per-class statistics. Channels 0-2
of each IMU are accelerometer-like; channels 3-5 are gyro-like. Same Gaussian
noise floor (sigma=0.1) on every class, added before zero-padding so pads stay
exactly zero.
"""


def _generate_one_attempt(params, rng):
    """Generate a single (T_MAX, 30) attempt for the given class params."""
    length = params["length"]
    accel_std = params["accel_std"]
    gyro_std = params["gyro_std"]
    freq_hz = params["freq_hz"]

    # Start with zeros so the pad region (length:T_MAX) stays exactly zero.
    x = np.zeros((T_MAX, N_CHANNELS), dtype=np.float32)

    # Time vector for the active portion of the attempt.
    t = np.arange(length) / FS  # seconds

    # Class-specific sinusoidal component shared across channels but with a
    # random phase per channel so the network sees coordinated-but-not-identical
    # oscillation across IMUs.
    phases = rng.uniform(0, 2 * np.pi, size=N_CHANNELS).astype(np.float32)
    sine = SINE_AMPLITUDE * np.sin(
        2 * np.pi * freq_hz * t[:, None] + phases[None, :]
    ).astype(np.float32)

    # Structured random walk in accel channels scaled to accel_std, plus the
    # same idea scaled to gyro_std for gyro channels. This gives the variance
    # signal its raw magnitude.
    for imu in range(N_IMUS):
        base = imu * 6
        for axis in ACCEL_AXES:
            ch = base + axis
            x[:length, ch] = rng.normal(0.0, accel_std, size=length)
        for axis in GYRO_AXES:
            ch = base + axis
            x[:length, ch] = rng.normal(0.0, gyro_std, size=length)

    # Overlay the class-specific sinusoid on the active region only.
    x[:length, :] += sine

    # Gaussian noise floor, same sigma for all classes, applied only to the
    # active region. The zero-padded tail remains identically zero so the
    # transition index is an unambiguous duration cue.
    x[:length, :] += rng.normal(0.0, NOISE_STD, size=(length, N_CHANNELS)).astype(np.float32)

    return x

In [4]:
def generate_synthetic_imu(n_per_class, seed=42):
    """
    Generate a synthetic 3-class IMU dataset for APEX pipeline validation.

    Parameters
    ----------
    n_per_class : int
        Number of attempts per class. Total samples = 3 * n_per_class.
    seed : int
        RNG seed for reproducibility.

    Returns
    -------
    X : np.ndarray, shape (3*n_per_class, 4500, 30), dtype float32
    y : np.ndarray, shape (3*n_per_class,),           dtype int64
        Class labels: 0=easy, 1=medium, 2=hard.
    """
    rng = np.random.default_rng(seed)

    n_total = 3 * n_per_class
    X = np.zeros((n_total, T_MAX, N_CHANNELS), dtype=np.float32)
    y = np.zeros(n_total, dtype=np.int64)

    idx = 0
    for class_label in range(3):
        params = CLASS_PARAMS[class_label]
        for _ in range(n_per_class):
            X[idx] = _generate_one_attempt(params, rng)
            y[idx] = class_label
            idx += 1

    # Shuffle so class order is not an artifact in any downstream split.
    perm = rng.permutation(n_total)
    return X[perm], y[perm]

In [ ]:
# Quick sanity check: generate a small dataset and verify the design
# predictions hold on the raw tensors.
X, y = generate_synthetic_imu(n_per_class=50, seed=42)
print(f"X shape: {X.shape}, dtype: {X.dtype}")
print(f"y shape: {y.shape}, class counts: {np.bincount(y)}\n")

print("Per-class verification (should be monotonically increasing):")
print(f"{'class':<8}{'accel_std':<14}{'gyro_std':<14}{'nonzero_len':<14}")

for c in range(3):
    Xc = X[y == c]

    # Nonzero length = index of the last nonzero sample + 1, averaged.
    lengths = []
    for attempt in Xc:
        nz = np.any(attempt != 0, axis=1)
        lengths.append(nz.sum())
    accel_chs = [imu * 6 + ax for imu in range(N_IMUS) for ax in ACCEL_AXES]
    gyro_chs = [imu * 6 + ax for imu in range(N_IMUS) for ax in GYRO_AXES]

    # Compute std only over nonzero region to avoid the pad diluting it.
    accel_std = np.mean([
        Xc[i, : lengths[i], accel_chs].std() for i in range(len(Xc))])
    gyro_std = np.mean([
        Xc[i, : lengths[i], gyro_chs].std() for i in range(len(Xc))])
    
    mean_len = np.mean(lengths)
    print(f"{c:<8}{accel_std:<14.3f}{gyro_std:<14.3f}{mean_len:<14.0f}")

X shape: (150, 4500, 30), dtype: float32
y shape: (150,), class counts: [50 50 50]

Per-class verification (should be monotonically increasing):
class   accel_std     gyro_std      nonzero_len   
0       0.474         0.418         1500          
1       0.880         0.704         2500          
2       1.543         1.255         3800          
